# Launch Qiskit Metal and set the chip dimensions

In [1]:
#To start the program:
#- Import the qiskit metal package
#- Instantiate the "DesignPlanar" class
#- Launch the GUI with the above instance as its argument
import qiskit_metal as metal
design = metal.designs.DesignPlanar()
gui = metal.MetalGUI(design)

In [2]:
#We enable the overwrite so we can modify the design
design.overwrite_enabled = True

# Notes:
#  layer_start and layer_end indicate the allowable layer numbers
#  sample_holder_top’ and ‘sample_holder_bottom’ specify the top and bottom coordinates of the
# vacuum above and below the chip.
design.chips.main.size.size_x = '6.5mm'
design.chips.main.size.size_y = '6.5mm'
design.chips.main.size.size_z = '-500um'
design.chips.main.size.sample_holder_top = '800um'
design.chips.main.size.sample_holder_bottom = '1000um'
design.chips.main

{'material': 'silicon',
 'layer_start': '0',
 'layer_end': '2048',
 'size': {'center_x': '0.0mm',
  'center_y': '0.0mm',
  'center_z': '0.0mm',
  'size_x': '6.5mm',
  'size_y': '6.5mm',
  'size_z': '-500um',
  'sample_holder_top': '800um',
  'sample_holder_bottom': '1000um'}}

# Define launch-pads and feedline

In [3]:
# This time I import the "LaunchpadWirebondDrive" class which is different from the "LaunchpadWirebond" 
# because it has an "in" pin in addition to the "tie" pin. We will use the "in" pin later when using the lumped ports
# to obtain the loaded quality factor
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven

# - Import the"Dict" class (We need it to create the ports and the feedline)
from qiskit_metal import Dict

In [4]:
# Instantiate LaunchpadWirebondDirven as many times as required. 
#In this case we need two pads
port_1 = LaunchpadWirebondDriven(design,'port_1', options = dict(pos_x = '-2.3mm',
                                                           pos_y = '0mm',
                                                           orientation = '0',
                                                           pad_width = '600um',
                                                           pad_height = '130um',
                                                           taper_height = '350um',
                                                           lead_length = '10um',
                                                           trace_width = '10um',
                                                           trace_gap ='5um',
                                                           pad_gap = '200um'))

port_2 = LaunchpadWirebondDriven(design,'port_2', options = dict(pos_x = '2.3mm',
                                                           pos_y = '0mm',
                                                           orientation = '180',
                                                           pad_width = '600um',
                                                           pad_height = '130um',
                                                           taper_height = '350um',
                                                           lead_length = '10um',
                                                           trace_width = '10um',
                                                           trace_gap ='5um',
                                                           pad_gap = '200um'))


#To see the change in the GUI run the following commands
gui.rebuild()
gui.autoscale()

In [5]:
# Then I add the feedline CPW between port 1 and port 2:
# - Import the "RouteSmeander" class becasue I need a curve
# - Set start_straight and end_straight very long. In this case 5 mm
# - Set meander=Dict(spacing...) also long so that it doesn't make inner curves. In this case: 4500 um

from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight

feedline = RouteStraight(design,'feedline', options = dict(hfss_wire_bonds = False, # This doesn't include wire_bonds in the HFSS rendering
                                                           pin_inputs=Dict(start_pin=Dict(component = 'port_1', pin='tie'),
                                                           end_pin=Dict(component = 'port_2', pin ='tie')),
                                                      trace_width = '10um',
                                                      trace_gap = '5um'))
gui.rebuild()
gui.autoscale()

# Defining the resonators

In [6]:
# To create the resonators, we first need to create their start and end points, so we import two classes:
# - One for the couplers to the feed line
# - And another one to set the output of the resonator as an open or short circuit
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander

In [82]:
coupl_1 = CoupledLineTee(design,'coupl_1', options=dict(pos_x = '-1.8mm', 
                                                        pos_y = '0.0 mm', 
                                                        orientation = '180',
                                                        coupling_length = '100um',
                                                        down_length = '200um',
                                                        fillet = '40um',
                                                        coupling_space = '20um',
                                                        prime_gap = '5um',
                                                        second_width = '8um',
                                                        second_gap = '4um',
                                                        open_termination = True))

short_end_1 = ShortToGround(design,'short_end_1',options=Dict(pos_x='-1.55mm',
                                                           pos_y='1mm',
                                                           orientation='90',
                                                           termination_gap='4um',
                                                           gap = '4um',
                                                           width = '8um'))
res_1 = RouteMeander(design, 'res_1', options = dict(pin_inputs=Dict(start_pin=Dict(component = 'coupl_1',
                                                                                    pin='second_end'),
                                                                     end_pin=Dict(component='short_end_1',
                                                                                  pin='short')),
                                                     lead=Dict(start_straight='0.2mm',
                                                               end_straight='0.3mm'),
                                                     meander=Dict(spacing='150um',
                                                                  asymmetry='0um',
                                                                  prevent_short_edges = True),
                                                     total_length='1.43mm',
                                                     trace_width='8um',
                                                     fillet='40um',
                                                     trace_gap='4um'))
gui.rebuild()
gui.autoscale()

In [83]:
coupl_2 = CoupledLineTee(design,'coupl_2', options=dict(pos_x = '-1.35mm', 
                                                        pos_y = '0.0 mm', 
                                                        orientation = '0',
                                                        coupling_length = '100um',
                                                        down_length = '200um',
                                                        fillet = '40um',
                                                        coupling_space = '20um',
                                                        prime_gap = '5um',
                                                        second_width = '10um',
                                                        second_gap = '5um',
                                                        open_termination = True))

short_end_2 = ShortToGround(design,'short_end_2',options=Dict(pos_x='-1.6mm',
                                                           pos_y='-1mm',
                                                           orientation='270',
                                                           termination_gap='5um',
                                                           gap = '5um',
                                                           width = '8um'))
res_2 = RouteMeander(design, 'res_2', options = dict(pin_inputs=Dict(start_pin=Dict(component = 'coupl_2',
                                                                                    pin='second_end'),
                                                                     end_pin=Dict(component='short_end_2',
                                                                                  pin='short')),
                                                     lead=Dict(start_straight='0.2mm',
                                                               end_straight='0.3mm'),
                                                     meander=Dict(spacing='150um',
                                                                  asymmetry='0um',
                                                                  prevent_short_edges = True),
                                                     total_length='1.5mm',
                                                     trace_width='10um',
                                                     fillet='40um',
                                                     trace_gap='5um'))
gui.rebuild()
gui.autoscale()

In [84]:
coupl_3 = CoupledLineTee(design,'coupl_3', options=dict(pos_x = '-0.9mm', 
                                                        pos_y = '0.0 mm', 
                                                        orientation = '180',
                                                        coupling_length = '100um',
                                                        down_length = '200um',
                                                        fillet = '40um',
                                                        coupling_space = '20um',
                                                        prime_gap = '5um',
                                                        second_width = '12um',
                                                        second_gap = '6um',
                                                        open_termination = True))

short_end_3 = ShortToGround(design,'short_end_3',options=Dict(pos_x='-0.65mm',
                                                           pos_y='1.0mm',
                                                           orientation='90',
                                                           termination_gap='6um',
                                                           gap = '6um',
                                                           width = '12um'))
res_3 = RouteMeander(design, 'res_3', options = dict(pin_inputs=Dict(start_pin=Dict(component = 'coupl_3',
                                                                                    pin='second_end'),
                                                                     end_pin=Dict(component='short_end_3',
                                                                                  pin='short')),
                                                     lead=Dict(start_straight='0.2mm',
                                                               end_straight='0.4mm'),
                                                     meander=Dict(spacing='150um',
                                                                  asymmetry='0um',
                                                                  prevent_short_edges = True),
                                                     total_length='1.55mm',
                                                     trace_width='12um',
                                                     fillet='40um',
                                                     trace_gap='6um'))
gui.rebuild()
gui.autoscale()

In [85]:
coupl_4 = CoupledLineTee(design,'coupl_4', options=dict(pos_x = '-0.45mm', 
                                                        pos_y = '0.0 mm', 
                                                        orientation = '0',
                                                        coupling_length = '100um',
                                                        down_length = '200um',
                                                        fillet = '40um',
                                                        coupling_space = '20um',
                                                        prime_gap = '5um',
                                                        second_width = '14um',
                                                        second_gap = '7um',
                                                        open_termination = True))

short_end_4 = ShortToGround(design,'short_end_4',options=Dict(pos_x='-0.7mm',
                                                           pos_y='-1.0mm',
                                                           orientation='270',
                                                           termination_gap='7um',
                                                           gap = '7um',
                                                           width = '14um'))
res_4 = RouteMeander(design, 'res_4', options = dict(pin_inputs=Dict(start_pin=Dict(component = 'coupl_4',
                                                                                    pin='second_end'),
                                                                     end_pin=Dict(component='short_end_4',
                                                                                  pin='short')),
                                                     lead=Dict(start_straight='0.2mm',
                                                               end_straight='0.4mm'),
                                                     meander=Dict(spacing='150um',
                                                                  asymmetry='0um',
                                                                  prevent_short_edges = True),
                                                     total_length='1.6mm',
                                                     trace_width='14um',
                                                     fillet='40um',
                                                     trace_gap='7um'))
gui.rebuild()
gui.autoscale()

02:49PM 04s CRITICAL [_qt_message_handler]: line: 0, func: None(), file: None  WARNING: QWindowsNativeFileDialogBase::selectNameFilter: Invalid parameter '*.gds' not found in 'All Files (*)'.

02:49PM 04s CRITICAL [_qt_message_handler]: line: 0, func: None(), file: None  WARNING: QWindowsNativeFileDialogBase::selectNameFilter: Invalid parameter '*.gds' not found in 'All Files (*)'.

02:49PM 16s WARNING [_import_junction_gds_file]: Not able to find file:"../resources/Fake_Junctions.GDS".  Not used to replace junction. Checked directory:"C:\Users\jmoeid\resources".


In [71]:
coupl_5 = CoupledLineTee(design,'coupl_5', options=dict(pos_x = '0.0mm', 
                                                        pos_y = '0.0 mm', 
                                                        orientation = '180',
                                                        coupling_length = '100um',
                                                        down_length = '200um',
                                                        fillet = '40um',
                                                        coupling_space = '20um',
                                                        prime_gap = '5um',
                                                        second_width = '8um',
                                                        second_gap = '4um',
                                                        open_termination = True))

short_end_5 = ShortToGround(design,'short_end_5',options=Dict(pos_x='0.25mm',
                                                           pos_y='0.8mm',
                                                           orientation='90',
                                                           termination_gap='4um',
                                                           gap = '4um',
                                                           width = '8um'))
res_5 = RouteMeander(design, 'res_5', options = dict(pin_inputs=Dict(start_pin=Dict(component = 'coupl_5',
                                                                                    pin='second_end'),
                                                                     end_pin=Dict(component='short_end_5',
                                                                                  pin='short')),
                                                     lead=Dict(start_straight='0.2mm',
                                                               end_straight='0.3mm'),
                                                     meander=Dict(spacing='150um',
                                                                  asymmetry='0um',
                                                                  prevent_short_edges = True),
                                                     total_length='1.083mm',
                                                     trace_width='8um',
                                                     fillet='40um',
                                                     trace_gap='4um'))
gui.rebuild()
gui.autoscale()

In [72]:
coupl_6 = CoupledLineTee(design,'coupl_6', options=dict(pos_x = '0.45mm', 
                                                        pos_y = '0.0 mm', 
                                                        orientation = '0',
                                                        coupling_length = '100um',
                                                        down_length = '200um',
                                                        fillet = '40um',
                                                        coupling_space = '20um',
                                                        prime_gap = '5um',
                                                        second_width = '10um',
                                                        second_gap = '5um',
                                                        open_termination = True))

short_end_6 = ShortToGround(design,'short_end_6',options=Dict(pos_x='0.2mm',
                                                           pos_y='-0.8mm',
                                                           orientation='270',
                                                           termination_gap='5um',
                                                           gap = '5um',
                                                           width = '8um'))
res_6 = RouteMeander(design, 'res_6', options = dict(pin_inputs=Dict(start_pin=Dict(component = 'coupl_6',
                                                                                    pin='second_end'),
                                                                     end_pin=Dict(component='short_end_6',
                                                                                  pin='short')),
                                                     lead=Dict(start_straight='0.2mm',
                                                               end_straight='0.3mm'),
                                                     meander=Dict(spacing='150um',
                                                                  asymmetry='0um',
                                                                  prevent_short_edges = True),
                                                     total_length='1.151mm',
                                                     trace_width='10um',
                                                     fillet='40um',
                                                     trace_gap='5um'))
gui.rebuild()
gui.autoscale()

In [73]:
coupl_7 = CoupledLineTee(design,'coupl_7', options=dict(pos_x = '0.9mm', 
                                                        pos_y = '0.0 mm', 
                                                        orientation = '180',
                                                        coupling_length = '100um',
                                                        down_length = '200um',
                                                        fillet = '40um',
                                                        coupling_space = '20um',
                                                        prime_gap = '5um',
                                                        second_width = '12um',
                                                        second_gap = '6um',
                                                        open_termination = True))

short_end_7 = ShortToGround(design,'short_end_7',options=Dict(pos_x='1.15mm',
                                                           pos_y='0.9mm',
                                                           orientation='90',
                                                           termination_gap='6um',
                                                           gap = '6um',
                                                           width = '12um'))
res_7 = RouteMeander(design, 'res_7', options = dict(pin_inputs=Dict(start_pin=Dict(component = 'coupl_7',
                                                                                    pin='second_end'),
                                                                     end_pin=Dict(component='short_end_7',
                                                                                  pin='short')),
                                                     lead=Dict(start_straight='0.2mm',
                                                               end_straight='0.4mm'),
                                                     meander=Dict(spacing='150um',
                                                                  asymmetry='0um',
                                                                  prevent_short_edges = True),
                                                     total_length='1.202mm',
                                                     trace_width='12um',
                                                     fillet='40um',
                                                     trace_gap='6um'))
gui.rebuild()
gui.autoscale()

In [74]:
coupl_8 = CoupledLineTee(design,'coupl_8', options=dict(pos_x = '1.35mm', 
                                                        pos_y = '0.0 mm', 
                                                        orientation = '0',
                                                        coupling_length = '100um',
                                                        down_length = '200um',
                                                        fillet = '40um',
                                                        coupling_space = '20um',
                                                        prime_gap = '5um',
                                                        second_width = '14um',
                                                        second_gap = '7um',
                                                        open_termination = True))

short_end_4 = ShortToGround(design,'short_end_8',options=Dict(pos_x='1.10mm',
                                                           pos_y='-0.9mm',
                                                           orientation='270',
                                                           termination_gap='7um',
                                                           gap = '7um',
                                                           width = '14um'))
res_8 = RouteMeander(design, 'res_8', options = dict(pin_inputs=Dict(start_pin=Dict(component = 'coupl_8',
                                                                                    pin='second_end'),
                                                                     end_pin=Dict(component='short_end_8',
                                                                                  pin='short')),
                                                     lead=Dict(start_straight='0.2mm',
                                                               end_straight='0.4mm'),
                                                     meander=Dict(spacing='150um',
                                                                  asymmetry='0um',
                                                                  prevent_short_edges = True),
                                                     total_length='1.241mm',
                                                     trace_width='14um',
                                                     fillet='40um',
                                                     trace_gap='7um'))
gui.rebuild()
gui.autoscale()

# Delete components

In [75]:
# Use this routine anytime you want to remove a component from the design

design.delete_component('coupl_1')
design.delete_component('short_end_1')
design.delete_component('res_1')
design.delete_component('coupl_2')
design.delete_component('short_end_2')
design.delete_component('res_2')
design.delete_component('coupl_3')
design.delete_component('short_end_3')
design.delete_component('res_3')
design.delete_component('coupl_4')
design.delete_component('short_end_4')
design.delete_component('res_4')
gui.rebuild()
gui.autoscale()

INFO 02:08PM [__del__]: Disconnected from Ansys HFSS
INFO 02:08PM [__del__]: Disconnected from Ansys HFSS


In [77]:
hfss_render.stop()

True

In [78]:
#I now start the steps to load the design to HFSS and launch HFSS
#First we instantiate an hfss renderer and Then we use the .start() 
#function to launch the Ansys Electronics Desktop (AEDT), create a new 
#project,and connect it to this notebook. 
#If you're outside NYU: reboot your pc and connect to the NYU VPN first.
hfss_render = design.renderers.hfss
hfss_render.start()

INFO 02:09PM [connect_project]: Connecting to Ansys Desktop API...
INFO 02:09PM [load_ansys_project]: 	Opened Ansys App
INFO 02:09PM [load_ansys_project]: 	Opened Ansys Desktop v2025.1.0
INFO 02:09PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/MoeidJamalzadeh/Documents/Ansoft/
	Project:   Project3
INFO 02:09PM [connect_design]: No active design found (or error getting active design).
INFO 02:09PM [connect]: 	 Connected to project "Project3". No design detected


True

# Eigenmode Simulation:

In [79]:
#Next, use the .new_
# ansys_design() to create an HFSS design in Ansys EDT
#with eigenmode solution. This design gets linked to the previously 
#defined renderer. 
#The arguments of the function are the design name and the solution mode
#After runing this command, the HFSS design opens in the Ansys EDT
hfss_eigen = hfss_render.new_ansys_design("Q11.1TaCN7.0_HighFreq", 'eigenmode')

INFO 02:09PM [connect_design]: 	Opened active design
	Design:    Q11.1TaCN7.0_HighFreq [Solution type: Eigenmode]
WARNING 02:09PM [connect_setup]: 	No design setup detected.
WARNING 02:09PM [connect_setup]: 	Creating eigenmode default setup.
INFO 02:09PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)


In [80]:
#The .render_design() function renders the entire design in HFSS
#After running this command, the qiskit metal design is loaded in HFSS
#Arguments:
#-selection:
#-open_pins:
#-port_list: specify the location of lumped ports. Lumped ports are connected to the "in" pin of the given port
#-box_plus_buffer = False makes the render to ignore the default buffer from metal to the border of the chip and keeps 
# the size of the chip equal to that defined at the begining of the code

#For more information and other options visit: 
#https://qiskit.org/documentation/metal/tut/3-Renderers/3.3-Render-your-design-to-Ansys.html

hfss_render.render_design(selection = [],
                          open_pins=[],
                          port_list=[('port_1','in', 50),('port_2', 'in', 50)],
                          box_plus_buffer = False)

In [81]:
# set simulation parameters
hfss_render.add_eigenmode_setup(name="EigenSetup", 
                                n_modes =10, 
                                max_passes = 20,
                                delta_f = 2,
                                min_converged = 5.5)

In [27]:
# run the simulation
hfss_render.analyze_setup("EigenSetup")

INFO 06:03PM [get_setup]: 	Opened setup `EigenSetup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 06:03PM [analyze]: Analyzing setup EigenSetup


com_error: (-2147352567, 'Exception occurred.', (0, None, None, None, 0, -2147024349), None)

In [24]:
#render EM fields to identify which mode belongs to each resonator
hfss_render.plot_fields('main')

In [25]:
from qiskit_metal.analyses.simulation import ScatteringImpedanceSim

In [26]:
em1 = ScatteringImpedanceSim(design, "hfss")
hfss = em1.renderer

In [27]:
#Create a new drivenmodal workspace inside the same HFSS project
hfss.activate_ansys_design("TL_Launchpads_and_Reson", 'drivenmodal')

11:43AM 48s WARNING [activate_ansys_design]: The design_name=TL_Launchpads_and_Reson was not in active project.  Designs in active project are: 
['ResonEigen'].  A new design will be added to the project.  
INFO 11:43AM [connect_design]: 	Opened active design
	Design:    TL_Launchpads_and_Reson [Solution type: HFSS Hybrid Modal Network]
WARNING 11:43AM [connect_setup]: 	No design setup detected.
WARNING 11:43AM [connect_setup]: 	Creating driven modal default setup.
INFO 11:43AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssDMSetup'>)


In [28]:
hfss_render.render_design(selection = [],
                          open_pins=[],
                          port_list=[('port_1','in', 50),('port_2', 'in', 50)],
                          box_plus_buffer = False)

In [ ]:
hfss.add_sweep(setup_name="Setup",
               name="Sweep",
               start_ghz=5,
               stop_ghz=5.5,
               count=3001,
               type="Fast")

INFO 11:44AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssDMSetup'>)


In [30]:
hfss.analyze_sweep('Sweep', 'Setup')


INFO 11:45AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssDMSetup'>)
INFO 11:45AM [analyze]: Analyzing setup Setup : Sweep


# Important Step: Adding a mesh to S parameter simulation

First completely do the eigen mode simulation.

For running the S parameter you need to implement the mesh design of the eigen mode to the setup of the S parameter.

Follow these steps:

1- After running the 29th block (hfss.add_sweep ...), right click on the setup in the S parameter section.

2- Go to the properties, in the Advanced section, import the EigenSetup from the eigen mode simulation.

3- Run the sweep command from python. It will take longer time to simulate.

**########### Important notes and common errors:**


 make sure you are referring to a same project in importing the mesh, if it's from other project, make sure the geometrical structure of the projects are exactly the same.

Choose max passes and counts wisely! simulation could take too long without even crashing (11 hours tested!), so choose number in a way that simulation don't take forever. If it's needed, shorten the width and try many times.

If you faced {com ... } error, it's mostly because you haven't run one library right, or maybe you need to rerun it! yes, it sometimes happen. If none of them worked, close all the simulation apps, restart kernel, and run the simulation from start carefully.

Watch out for warnings in the previous blocks, they do matter. there must be no negative warning like (something is missing..., that project does not exist..., device didn't implement...). Even look at the warnings in HFSS or qiskit metal window.

Every order of running matters! If one is wrong, you must close the whole software and rerun the code.

If you are running several sweeps (or eigensetups) in one project, watch out for the names of the simulation. since the names are duplicate, the next ones would be for example (sweep1, sweep2 ...), Make sure you are running a right sweep in running command.  

Watch out for the variables' details! some variables are not defined in the code. You can find them in Qiskit metal vindow or HFSS window.
Always check your design, specially in the borders, to not have different lengths or widths. check it with qiskit.

If you face "Not defining adesign" error, follow the following steps:

        ○ File: Users\<user>\anaconda3\envs\pymetal23\Lib\site-packages\pyEPR\ansys.py
            § Line 716:
                □ It was: elif self.solution_type == "DrivenModal":
                □ I changed it to: elif self.solution_type == "HFSS Hybrid Modal Network":
    
        • File: Users\<user>\anaconda3\envs\pymetal23\Lib\site-packages\pyEPR\project_info.py
            § Line 326:
                □ It was: elif self.design.solution_type == 'DrivenModal':
                □ I changed it to: elif self.design.solution_type == 'HFSS Hybrid Modal Network':
    
That was what I needed to change, nothing more, and make sure that after these changes you import qiskit_metal to your code again so the changes take effect

Be aware of the function lower and uppercase. For example, it's not "DrivenModal", it's "drivenmodal". You can find these in the source code.

It is suggested to have 1 project and save it each time, and just the variable you want each time and run it in the HFSS, not python. Keeping it in same project would make it much faster.